# 🔍 Plant 2: Baseline Exploratory Data Analysis (Sandbox Framework)

This notebook serves as our raw experimental sandbox for Plant 2. Here, we execute data ingestion, inspect raw data schema structures, and map physical diurnal solar profiles along with cumulative asset yields to uncover embedded systemic anomalies.

In [ ]:
# ==============================================================================
# SECTION 1: GLOBAL LIBRARY INGESTION & CONFIGURATION
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

# Set clean whitegrid visualization parameters globally
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('ggplot')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

## 💾 Data Ingestion & Chronological Synchronization
We load the completely raw generation and meteorological sensor streams independently, enforcing unified pandas datetime indexing keys across the assets.

In [ ]:
# ==============================================================================
# SECTION 2: RAW DATA INGESTION & DATETIME STRUCTURE ENFORCEMENT
# ==============================================================================

# Define dynamic local file routing (fallback safely if running locally or repo root)
gen_path = '../data/raw/Plant_2_Generation_Data.csv' if os.path.exists('../data/raw/Plant_2_Generation_Data.csv') else 'Plant_2_Generation_Data.csv'
weather_path = '../data/raw/Plant_2_Weather_Sensor_Data.csv' if os.path.exists('../data/raw/Plant_2_Weather_Sensor_Data.csv') else 'Plant_2_Weather_Sensor_Data.csv'

# Load records safely
gen_p2 = pd.read_csv(gen_path)
weather_p2 = pd.read_csv(weather_path)

# Enforce uniform datetime formatting structures
gen_p2['DATE_TIME'] = pd.to_datetime(gen_p2['DATE_TIME'])
weather_p2['DATE_TIME'] = pd.to_datetime(weather_p2['DATE_TIME'])

# Isolate clean string formatting trackers for prospective visualization subplots
gen_p2['DATE_STR'] = gen_p2['DATE_TIME'].dt.strftime('%Y-%m-%d')
gen_p2['TIME_STR'] = gen_p2['DATE_TIME'].dt.strftime('%H:%M')

## 🗂️ Metadata & Feature Schema Auditing
Before visualizing arrays, we output a structural text profile report to audit baseline sample availability, raw feature features, and cross-sectional data patterns.

In [ ]:
# ==============================================================================
# SECTION 3: METADATA & FEATURE STRUCTURE REPORT
# ==============================================================================
print("==============================================================================")
print(" 1. METADATA & FEATURE STRUCTURE REPORT")
print("==============================================================================")
print("\n[ISOLATED DATASET A: GENERATION DATA]")
print(f"Total Database Rows : {gen_p2.shape[0]} rows")
print(f"Available Features  : {list(gen_p2.columns)}")
print("\n--- Raw Head Sample ---")
print(gen_p2.head(3).to_string(index=False))

print("\n" + "="*80)
print("\n[ISOLATED DATASET B: WEATHER SENSOR DATA]")
print(f"Total Database Rows : {weather_p2.shape[0]} rows")
print(f"Available Features  : {list(weather_p2.columns)}")
print("\n--- Raw Head Sample ---")
print(weather_p2.head(3).to_string(index=False))
print("\n" + "="*80 + "\n")

## ☀️ Overlaid Diurnal Power Profile Analysis
To verify physical scaling parameters and chronological data alignment, we sample an arbitrary operational day (`2020-05-25`) and aggregate the plant's cross-sectional mean generation track.

In [ ]:
# ==============================================================================
# SECTION 4: DIURNAL POWER PROFILE VISUALIZATION
# ==============================================================================

# Establish standard target profile baseline date
target_sample_date = '2020-05-25'
single_day_gen = gen_p2[gen_p2['DATE_STR'] == target_sample_date].sort_values('DATE_TIME')

# Aggregate by exact timestamp keys to smooth out localized inverter dropouts
single_day_profile = single_day_gen.groupby('TIME_STR')[['DC_POWER', 'AC_POWER']].mean().reset_index()

plt.figure(figsize=(15, 6))

# Plot raw current generation scaling metrics symmetrically
plt.plot(single_day_profile['TIME_STR'], single_day_profile['DC_POWER'], 
         color='crimson', lw=2.5, marker='o', ms=4, label='DC Power Output (Panel Side)')
plt.plot(single_day_profile['TIME_STR'], single_day_profile['AC_POWER'], 
         color='darkblue', lw=2, linestyle='--', label='AC Power Output (Inverter Side)')

# Secure chronological tick density allocation symmetrically (Show tick every 4 hours)
ax1 = plt.gca()
current_ticks = ax1.get_xticks()
ax1.set_xticks(current_ticks[::16])
plt.xticks(rotation=45)

plt.title(f"Plant 2: Overlaid Chronological Diurnal Power Profile ({target_sample_date})", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day", fontsize=11)
plt.ylabel("Power Metrics (kW)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

## 📊 Asset-Level Cumulative Yield & Lifetime Dashboard
To investigate the integrity of the integrated registry counters, we isolate a specific sub-asset inverter node, capture its maximum daily boundaries, and construct a dual-axis operational yield visualization dashboard.

In [ ]:
# ==============================================================================
# SECTION 5: CUMULATIVE YIELD FIXED DASHBOARD (WARNING RESOLVED)
# ==============================================================================

# Target a singular baseline hardware sub-asset string
target_asset = gen_p2['SOURCE_KEY'].unique()[0]
asset_df = gen_p2[gen_p2['SOURCE_KEY'] == target_asset].sort_values('DATE_TIME')

# Compile maximum end-of-day timeline aggregations safely
daily_summary = asset_df.groupby('DATE_STR').agg({
    'DAILY_YIELD': 'max',
    'TOTAL_YIELD': 'max'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(16, 6))

# Left Axis (Y1): Plot Daily Volume Track as Chronological Bars
ax1.bar(daily_summary['DATE_STR'], daily_summary['DAILY_YIELD'], 
        color='orange', edgecolor='darkorange', alpha=0.7, label='Daily Accumulated Yield (kWh)')
ax1.set_xlabel("Calendar Operation Date", fontsize=11)
ax1.set_ylabel("Daily Energy Volume (kWh)", color='darkorange', fontsize=11)
ax1.tick_params(axis='y', labelcolor='darkorange')

# FIX: Establish explicit tick array indices BEFORE attempting text mapping to block UserWarning diagnostics
x_indices = np.arange(len(daily_summary['DATE_STR']))
ax1.set_xticks(x_indices)
ax1.set_xticklabels(daily_summary['DATE_STR'], rotation=90, fontsize=9)
ax1.grid(True, axis='y', linestyle='--', alpha=0.3)

# Right Axis (Y2): Overlay Lifetime Cumulative Growth Lines symmetrically
ax2 = ax1.twinx()
ax2.plot(daily_summary['DATE_STR'], daily_summary['TOTAL_YIELD'], 
         color='royalblue', lw=3, marker='o', ms=5, label='Lifetime Total Yield (Cumulative kWh)')
ax2.set_ylabel("Lifetime Total Volume (kWh)", color='royalblue', fontsize=11)
ax2.tick_params(axis='y', labelcolor='royalblue')

plt.title(f"Plant 2: Cumulative Yield Dashboard for Asset [{target_asset}]", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

# 📈 Plant 2: Meteorological Sensor Distribution & Summary Statistics

To build a high-fidelity machine learning model, we must thoroughly understand the atmospheric independent variables ($X$). This section computes the numerical summaries and maps the statistical distributions (Histograms & Boxplots) for ambient air temperature, panel surface module temperature, and solar irradiation intensity.

### Key Data Science Focus Points:
* **Thermal Disparity:** Comparing the structural variance between the surrounding air (Ambient) and the physical panel surface heat (Module) to understand thermal efficiency degradation rules.
* **Nocturnal Mass Distribution:** Auditing the heavy statistical mass at the $0.0\ W/m^2$ boundary inside the Solar Irradiation vector, which reflects night-time states and requires strict non-linear splitting logic.

In [ ]:
# ==============================================================================
# SECTION 6: ISOLATED DESCRIPTIVE STATISTICS (NUMERICAL VERIFICATION)
# ==============================================================================

print("\n" + "="*80)
print(" 2. WEATHER SENSOR DATA - SUMMARY STATISTICS")
print("==============================================================================")

# Target vectors for atmospheric data profiles
weather_metrics = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
print(weather_p2[weather_metrics].describe().to_string())

print("\n" + "="*80 + "\n")

## 📊 Frequency Density & Statistical Spread Canvas
We plot a $3 \times 2$ matrix combining Histograms (with Kernel Density Estimate curves) and Boxplots to cross-examine frequency density profiles alongside outer statistical outliers.

In [ ]:
# ==============================================================================
# SECTION 7: METRIC DISTRIBUTION VISUALIZATION (HISTOGRAMS & BOXPLOTS)
# ==============================================================================

# Set up a structured plotting grid for the three primary weather sensors
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# Define color themes for distinct physical properties
colors = {'AMBIENT': 'teal', 'MODULE': 'darkred', 'IRRADIATION': 'gold'}


# --- FEATURE A: AMBIENT TEMPERATURE (Surrounding Air) ---
# Left: Histogram with Kernel Density Estimate (KDE) curve
sns.histplot(weather_p2['AMBIENT_TEMPERATURE'], bins=50, kde=True, 
             color=colors['AMBIENT'], ax=axes[0, 0], edgecolor='black')
axes[0, 0].set_title("Ambient Temperature - Frequency Density", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Temperature (°C)")

# Right: Boxplot to audit median limits and statistical outliers
sns.boxplot(x=weather_p2['AMBIENT_TEMPERATURE'], color=colors['AMBIENT'], ax=axes[0, 1])
axes[0, 1].set_title("Ambient Temperature - Statistical Spread Boxplot", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Temperature (°C)")


# --- FEATURE B: MODULE TEMPERATURE (Panel Surface Heat) ---
# Left: Histogram to isolate operational thermal clusters
sns.histplot(weather_p2['MODULE_TEMPERATURE'], bins=50, kde=True, 
             color=colors['MODULE'], ax=axes[1, 0], edgecolor='black')
axes[1, 0].set_title("Module Temperature - Frequency Density", fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel("Temperature (°C)") # FIX: Corrected axis index from [1, 1] to [1, 0] to map labels safely

# Right: Boxplot to check upper extreme heat variations
sns.boxplot(x=weather_p2['MODULE_TEMPERATURE'], color=colors['MODULE'], ax=axes[1, 1])
axes[1, 1].set_title("Module Temperature - Statistical Spread Boxplot", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Temperature (°C)")


# --- FEATURE C: SOLAR IRRADIATION (Sunlight Intensity) ---
# Left: Histogram mapping daytime transitions vs nocturnal zero-states
sns.histplot(weather_p2['IRRADIATION'], bins=50, kde=True, 
             color=colors['IRRADIATION'], ax=axes[2, 0], edgecolor='black')
axes[2, 0].set_title("Solar Irradiation - Frequency Density", fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel("Irradiation ($W/m^2$)")

# Right: Boxplot highlighting operational sunlight distributions
sns.boxplot(x=weather_p2['IRRADIATION'], color=colors['IRRADIATION'], ax=axes[2, 1])
axes[2, 1].set_title("Solar Irradiation - Statistical Spread Boxplot", fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel("Irradiation ($W/m^2$)")

plt.tight_layout()
plt.show()

# 📉 Plant 2: 34-Day Inverter Conversion Efficiency Timeline Analysis

To diagnostic mechanical degradation and operational underperformance, we extract the physical **AC/DC Conversion Efficiency Ratio**. This ratio represents the efficiency of the inverter hardware as it translates Direct Current ($DC$) from the panels into usable Alternating Current ($AC$) waves.

### Domain-Driven Engineering Safeguards:
1. **Diurnal Windowing Constraint:** We restrict our analytical boundary strictly between `09:00 AM` and `04:00 PM`. This isolates peak operational daylight hours and eliminates numerical instability caused by solar zenith angle drops.
2. **Zero-Division Censorship:** We explicitly censor records where $DC\_POWER \le 0$ to prevent computational infinity errors ($\infty$) during ratio generation, guaranteeing a stable baseline footprint.

In [ ]:
# ==============================================================================
# SECTION 8: 34-DAY INVERTER EFFICIENCY TIMELINE TRAJECTORY
# ==============================================================================

# Ensure raw dataset is active in the local memory namespace
if 'gen_p2' not in locals():
    gen_path = '../data/raw/Plant_2_Generation_Data.csv' if os.path.exists('../data/raw/Plant_2_Generation_Data.csv') else 'Plant_2_Generation_Data.csv'
    gen_p2 = pd.read_csv(gen_path)
    gen_p2['DATE_TIME'] = pd.to_datetime(gen_p2['DATE_TIME'])
    gen_p2['DATE_STR'] = gen_p2['DATE_TIME'].dt.strftime('%Y-%m-%d')
    gen_p2['TIME_STR'] = gen_p2['DATE_TIME'].dt.strftime('%H:%M')

# 1. FILTERING SAFEGUARD: Isolate peak daylight hours to eliminate zero-division at night
# We filter strictly between 09:00 AM and 04:00 PM when generation is stable
daylight_df = gen_p2[(gen_p2['TIME_STR'] >= '09:00') & (gen_p2['TIME_STR'] <= '16:00')].copy()

# Ensure we drop any random rows where DC_POWER is exactly 0 to avoid zero-division errors
daylight_df = daylight_df[daylight_df['DC_POWER'] > 0]

# 2. CALCULATE EFFICIENCY: Apply the physical conversion ratio rule (AC / DC)
daylight_df['EFFICIENCY'] = daylight_df['AC_POWER'] / daylight_df['DC_POWER']

# 3. CHRONOLOGICAL AGGREGATION: Compute the mean efficiency for the entire plant per day
daily_efficiency = daylight_df.groupby('DATE_STR')['EFFICIENCY'].mean().reset_index()

# 4. PLOT TIMELINE TREND LINE
plt.figure(figsize=(16, 6))

# Plot the 34-day operational trajectory
plt.plot(daily_efficiency['DATE_STR'], daily_efficiency['EFFICIENCY'], 
         color='forestgreen', lw=3, marker='o', ms=6, label='Mean Plant Conversion Efficiency (AC/DC)')

# Set strict Y-axis limits matching real-world physical boundaries
plt.ylim(0.0, 1.1)  # Efficiency scales between 0% and 110% (captures scaling characteristics)

# FIX: Anchor the explicit tick array locator before setting label properties to block UserWarning diagnostics
x_indices = np.arange(len(daily_efficiency['DATE_STR']))
plt.gca().set_xticks(x_indices)
plt.gca().set_xticklabels(daily_efficiency['DATE_STR'], rotation=90, fontsize=9)

plt.title("Plant 2: 34-Day Continuous Inverter Efficiency Trajectory Trend Line", fontsize=14, fontweight='bold')
plt.xlabel("Calendar Operation Date", fontsize=11)
plt.ylabel("Conversion Efficiency Ratio (AC / DC)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='lower left', fontsize=11)
plt.tight_layout()
plt.show()

# 🔍 Plant 2: Missing Value (Null/NaN) Operational Data Audit

Before advancing to structural feature engineering and model training, a rigorous data management architecture requires an **Operational Data Integrity Sign-Off**. This section systematically scans both the Generation Telemetry Matrix and the Weather Sensor Matrix to compute absolute missing value counts and their respective percentage weights.

### Data Engineering Significance:
* **Imputation Strategy Rationale:** Identifying whether missing fields require mathematical imputation (such as forward-filling or median interpolation) or structural row censorship.
* **Pipeline Safety Sign-Off:** Confirming the data stream is 100% complete prevents structural arrays from throwing runtime exceptions during LightGBM tensor conversion.

In [ ]:
# ==============================================================================
# SECTION 9: MISSING VALUE (NULL/NAN) OPERATIONAL DATA AUDIT
# ==============================================================================

# 1. Enforce active memory allocation for completely raw data assets (relying on centralized paths)
if 'gen_p2' not in locals() or 'weather_p2' not in locals():
    gen_path = '../data/raw/Plant_2_Generation_Data.csv' if os.path.exists('../data/raw/Plant_2_Generation_Data.csv') else 'Plant_2_Generation_Data.csv'
    weather_path = '../data/raw/Plant_2_Weather_Sensor_Data.csv' if os.path.exists('../data/raw/Plant_2_Weather_Sensor_Data.csv') else 'Plant_2_Weather_Sensor_Data.csv'
    
    gen_p2 = pd.read_csv(gen_path)
    weather_p2 = pd.read_csv(weather_path)


# ------------------------------------------------------------------------------
# AUDIT BLOCK A: GENERATION DATA MATRIX
# ------------------------------------------------------------------------------
# Calculate absolute null counts and map them to their percentage weights
gen_null_counts = gen_p2.isnull().sum()
gen_null_pcts   = (gen_p2.isnull().sum() / len(gen_p2)) * 100

gen_audit_table = pd.DataFrame({
    'Feature_Name': gen_null_counts.index,
    'Total_Null_Rows': gen_null_counts.values,
    'Missing_Ratio(%)': gen_null_pcts.values
})

print("==============================================================================")
print(" AUDIT REPORT A: PLANT 2 GENERATION DATASET INTEGRITY SIGN-OFF")
print("==============================================================================")
print(gen_audit_table.to_string(index=False, formatters={'Missing_Ratio(%)': '{:.4f}%'.format}))
print(f"--> Total Rows Scanned: {len(gen_p2):,} rows")
print("==============================================================================\n")


# ------------------------------------------------------------------------------
# AUDIT BLOCK B: WEATHER SENSOR DATA MATRIX
# ------------------------------------------------------------------------------
# Calculate absolute null counts and map them to their percentage weights
weather_null_counts = weather_p2.isnull().sum()
weather_null_pcts   = (weather_p2.isnull().sum() / len(weather_p2)) * 100

weather_audit_table = pd.DataFrame({
    'Feature_Name': weather_null_counts.index,
    'Total_Null_Rows': weather_null_counts.values,
    'Missing_Ratio(%)': weather_null_pcts.values
})

print("==============================================================================")
print(" AUDIT REPORT B: PLANT 2 WEATHER SENSOR DATASET INTEGRITY SIGN-OFF")
print("==============================================================================")
print(weather_audit_table.to_string(index=False, formatters={'Missing_Ratio(%)': '{:.4f}%'.format}))
print(f"--> Total Rows Scanned: {len(weather_p2):,} rows")
print("==============================================================================")

# 🚨 Plant 2: Advanced EDA - Hardware Register Anomalies & Midnight Reset Defects

This section delivers empirical verification of the core mechanical and logging failures embedded within the Plant 2 dataset. By cross-examining micro-telemetry tracking, we expose severe structural defects that justify our subsequent machine learning features constraints.

### The Two Empirical Proofs:
1. **Midnight Registry Cache Leak (Proof 1):** Proving that individual inverter counters fail to flush their cumulative tracking metrics at `00:00 AM`, carrying over arbitrary legacy yields (exceeding 9,425 kWh) directly into the next day's dawn metrics.
2. **Sub-Asset Capacity Drift & Performance Slack (Proof 2):** Constructing midday distribution footprints to track localized equipment imbalances across all 22 inverters under identical macro weather metrics.

In [ ]:
# ==============================================================================
# SECTION 10: PROOF 1 - MIDNIGHT RESET BUG AUDIT DETECTION
# ==============================================================================

# Ensure raw dataset is active in the local memory namespace
if 'gen_p2' not in locals():
    gen_path = '../data/raw/Plant_2_Generation_Data.csv' if os.path.exists('../data/raw/Plant_2_Generation_Data.csv') else 'Plant_2_Generation_Data.csv'
    gen_p2 = pd.read_csv(gen_path)
    gen_p2['DATE_TIME'] = pd.to_datetime(gen_p2['DATE_TIME'])
    gen_p2['DATE_STR'] = gen_p2['DATE_TIME'].dt.strftime('%Y-%m-%d')
    gen_p2['TIME_STR'] = gen_p2['DATE_TIME'].dt.strftime('%H:%M')

# Extract midnight entries (00:00 AM) across all unique inverter assets
midnight_df = gen_p2[gen_p2['TIME_STR'] == '00:00'].sort_values(['DATE_STR', 'SOURCE_KEY'])

print("==============================================================================")
print(" CRITICAL AUDIT 1: MIDNIGHT RESET FAILURES DETECTED")
print("==============================================================================")
print("Notice how some inverters carry over thousands of kWh instead of resetting to 0.00:")
print(midnight_df[['DATE_STR', 'SOURCE_KEY', 'DAILY_YIELD']].head(6).to_string(index=False))
print("==============================================================================")

## 📊 Sub-Asset Midday Power Spread Distribution
To isolate asset performance drift, we capture continuous peaks between `11:00 AM` and `01:00 PM` across all sub-inverters, checking for operational slack or systemic underperformance markers.

In [ ]:
# ==============================================================================
# SECTION 11: PROOF 2 - INVERTER PERFORMANCE DISPARITY (WARNING RESOLVED)
# ==============================================================================

# Filter strictly for peak daylight hours to observe active generation characteristics
peak_daylight = gen_p2[(gen_p2['TIME_STR'] >= '11:00') & (gen_p2['TIME_STR'] <= '13:00')]

plt.figure(figsize=(16, 7))

# FIX: Explicitly map x to hue and set legend=False to fully block seaborn v0.14+ deprecation warnings
sns.boxplot(
    data=peak_daylight.sort_values('SOURCE_KEY'), 
    x='SOURCE_KEY', 
    y='DC_POWER', 
    hue='SOURCE_KEY',
    palette='viridis',
    legend=False
)

plt.xticks(rotation=90, fontsize=9)
plt.title("Plant 2: Midday DC Power Output Distribution Across Individual Inverters (11:00 - 13:00)", fontsize=14, fontweight='bold')
plt.xlabel("Unique Inverter Asset Identifiers (SOURCE_KEY)", fontsize=11)
plt.ylabel("DC Power Output (kW)", fontsize=11)
plt.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# 📊 Plant 2: 34-Day Multi-Facet Module Temperature Diurnal Scatter Analysis

To evaluate chronological consistency and audit structural gaps across the entire monitoring lifespan, we construct a **Multi-Facet Subplot Grid ($6 \times 6$)**. This matrix maps the diurnal panel surface temperature profile ($Module\_Temperature$ vs $Time\_of\_Day$) for each of the 34 operational calendar days independently.

### Analytical and Diagnostic Rationale:
* **Chronological Anomaly Screening:** Isolating daily thermal curves to immediately spot macro meteorological disruptions, sensor dropouts, or communication freeze states.
* **Feature Space Distribution Validation:** Confirming that the physical bell-shaped heat distribution matches the expected solar tracking curve symmetrically across all evaluation partitions.

In [ ]:
# ==============================================================================
# SECTION 12: 34-DAY MULTI-FACET MODULE TEMPERATURE DIURNAL SCATTER
# ==============================================================================

# Ensure raw dataset is active in the local memory namespace
if 'weather_p2' not in locals():
    weather_path = '../data/raw/Plant_2_Weather_Sensor_Data.csv' if os.path.exists('../data/raw/Plant_2_Weather_Sensor_Data.csv') else 'Plant_2_Weather_Sensor_Data.csv'
    weather_p2 = pd.read_csv(weather_path)
    weather_p2['DATE_TIME'] = pd.to_datetime(weather_p2['DATE_TIME'])

# Construct tracking indicators and sorting elements
df_mod = weather_p2.copy()
df_mod['DATE_STR'] = df_mod['DATE_TIME'].dt.strftime('%Y-%m-%d')
df_mod['TIME_MINUTES'] = df_mod['DATE_TIME'].dt.hour * 60 + df_mod['DATE_TIME'].dt.minute

# Extract all distinct calendar days sorted chronologically
unique_days = sorted(df_mod['DATE_STR'].unique())

# Initialize a 6x6 grid subplot framework to cleanly encompass 34 targets
fig, axes = plt.subplots(6, 6, figsize=(22, 20), sharex=True, sharey=True)
axes_flat = axes.flatten()

for i, day in enumerate(unique_days):
    day_data = df_mod[df_mod['DATE_STR'] == day]
    ax = axes_flat[i]
    
    # Scatter plot tracking the panel heat development path across 24 hours
    ax.scatter(day_data['TIME_MINUTES'], day_data['MODULE_TEMPERATURE'], 
               color='darkred', alpha=0.6, s=8)
    
    # Format localized sub-graph headers
    ax.set_title(day, fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # FIX: Explicitly set the ticks locator array to ensure compatibility with sharex constraints
    ax.set_xticks([360, 720, 1080])
    ax.set_xticklabels(['06:00', '12:00', '18:00'], fontsize=8)

# Disable empty unused grid plots remaining at the lower right edge
for j in range(len(unique_days), len(axes_flat)):
    fig.delaxes(axes_flat[j])

# Configure unified super-labels across the layout canvas
fig.suptitle("Plant 2: 34-Day Isolated Grids of Module Temperature vs Time of Day", fontsize=18, fontweight='bold', y=0.98)
fig.supxlabel("Time of Day (24-Hour Cycle)", fontsize=14)
fig.supylabel("Module Temperature (°C)", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# 📊 Plant 2: 34-Day Multi-Facet Ambient Temperature Diurnal Scatter Analysis

To cross-examine the thermal mechanics of the asset alongside the previous panel heat matrices, we construct an identical **Multi-Facet Subplot Grid ($6 \times 6$)** for the surrounding air temperature ($Ambient\_Temperature$ vs $Time\_of\_Day$) across the 34-day monitoring timeline.

### Analytical and Diagnostic Rationale:
* **Thermal Lag and Inversion Diagnostics:** Analyzing the micro-climate delta between raw atmospheric air heat (Teal profiles) and panel surface heat (Dark Red profiles) to model potential thermal efficiency saturation thresholds.
* **Atmospheric Consistency Check:** Detecting sudden weather fronts, localized sensor clipping states, or missing evening blocks that could skew spatial features inside our predictive architecture.

In [ ]:
# ==============================================================================
# SECTION 13: 34-DAY MULTI-FACET AMBIENT TEMPERATURE DIURNAL SCATTER
# ==============================================================================

# Ensure raw dataset is active in the local memory namespace
if 'weather_p2' not in locals():
    weather_path = '../data/raw/Plant_2_Weather_Sensor_Data.csv' if os.path.exists('../data/raw/Plant_2_Weather_Sensor_Data.csv') else 'Plant_2_Weather_Sensor_Data.csv'
    weather_p2 = pd.read_csv(weather_path)
    weather_p2['DATE_TIME'] = pd.to_datetime(weather_p2['DATE_TIME'])

# Construct tracking indicators and sorting elements
df_amb = weather_p2.copy()
df_amb['DATE_STR'] = df_amb['DATE_TIME'].dt.strftime('%Y-%m-%d')
df_amb['TIME_MINUTES'] = df_amb['DATE_TIME'].dt.hour * 60 + df_amb['DATE_TIME'].dt.minute

# Extract all distinct calendar days sorted chronologically
unique_days = sorted(df_amb['DATE_STR'].unique())

# Initialize a 6x6 grid subplot framework to cleanly encompass 34 targets
fig, axes = plt.subplots(6, 6, figsize=(22, 20), sharex=True, sharey=True)
axes_flat = axes.flatten() # FIX: Isolate flattened variable to protect the original dimensional framework

for i, day in enumerate(unique_days):
    day_data = df_amb[df_amb['DATE_STR'] == day]
    ax = axes_flat[i]
    
    # Scatter plot tracking the ambient air thermal changes across 24 hours
    ax.scatter(day_data['TIME_MINUTES'], day_data['AMBIENT_TEMPERATURE'], 
               color='teal', alpha=0.6, s=8)
    
    # Format localized sub-graph headers
    ax.set_title(day, fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # Explicitly set the ticks locator array to ensure compatibility with sharex constraints
    ax.set_xticks([360, 720, 1080])
    ax.set_xticklabels(['06:00', '12:00', '18:00'], fontsize=8)

# Disable empty unused grid plots remaining at the lower right edge safely
for j in range(len(unique_days), len(axes_flat)):
    fig.delaxes(axes_flat[j])

# Configure unified super-labels across the layout canvas
fig.suptitle("Plant 2: 34-Day Isolated Grids of Ambient Temperature vs Time of Day", fontsize=18, fontweight='bold', y=0.98)
fig.supxlabel("Time of Day (24-Hour Cycle)", fontsize=14)
fig.supylabel("Ambient Temperature (°C)", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# 🚨 Plant 2: Deep Empirical Audit - Midnight Registry Cache Leak Proof

This section delivers our absolute **Technical Flashpoint**. While previous checkpoints flagged individual row anomalies, we now isolate the continuous 24-hour chronological timeline across three distinct inverter assets on `2020-05-15`. 

### The Strategic Blueprint for Visual Proof:
* **The Control Baseline (Healthy Asset):** Tracking an inverter that achieves a pristine $0.00\ kWh$ reset at midnight, mapping the theoretical ground-truth conversion arc.
* **The Experimental Anomalies (Bugged Assets):** Plotting two independent faulty nodes to expose a **Floating Initial Intercept Effect**. We visualize how they carry thousands of historical $kWh$ over the midnight boundary, creating a distorted step-function pattern that would cause standard regression engines to fail.

In [ ]:
# ==============================================================================
# SECTION 14: PROOF 3 - MIDNIGHT FLASHPOINT EVIDENCE MATRIX
# ==============================================================================

# Ensure raw dataset is active in the local memory namespace
if 'gen_p2' not in locals():
    gen_path = '../data/raw/Plant_2_Generation_Data.csv' if os.path.exists('../data/raw/Plant_2_Generation_Data.csv') else 'Plant_2_Generation_Data.csv'
    gen_p2 = pd.read_csv(gen_path)
    gen_p2['DATE_TIME'] = pd.to_datetime(gen_p2['DATE_TIME'])
    gen_p2['DATE_STR'] = gen_p2['DATE_TIME'].dt.strftime('%Y-%m-%d')
    gen_p2['TIME_STR'] = gen_p2['DATE_TIME'].dt.strftime('%H:%M')

# Isolate entries at exactly midnight on the very first day (May 15, 2020)
target_test_date = '2020-05-15'
midnight_flashpoint = gen_p2[
    (gen_p2['DATE_STR'] == target_test_date) & 
    (gen_p2['TIME_STR'] == '00:00')
].sort_values('SOURCE_KEY')

print("==============================================================================")
print(f" EVIDENCE TABLE: DAILY_YIELD RESET FAILURES AT MIDNIGHT ({target_test_date} 00:00 AM)")
print("==============================================================================")
print("Look at the highlighted anomalies where features carry over legacy metrics instead of 0.00:")
print(midnight_flashpoint[['DATE_STR', 'TIME_STR', 'SOURCE_KEY', 'DAILY_YIELD']].to_string(index=False, formatters={'DAILY_YIELD': '{:,.2f}'.format}))
print("==============================================================================")

## 📊 24-Hour Continuous Trajectory Audit (Staircase Execution Comparison)
We render the overlaid chronological timelines to isolate the mathematical shift. This visualizes how the corrupted registry cache creates an artificial upward parallel floating drift from dawn to dusk.

In [ ]:
# ==============================================================================
# SECTION 15: CHRONOLOGICAL ACCUMULATION TRAJECTORY COMPARISON (WARNING RESOLVED)
# ==============================================================================

plt.figure(figsize=(15, 7))

# Establish our control groups: one healthy asset, two distinct faulty nodes
target_inverters = {
    '81aHJ1q11NBPMrL': {'label': 'Healthy Asset (Perfect 0.0 Reset)', 'color': 'teal', 'ls': '-'},
    '4UPUqMRk7TRMgml': {'label': 'Bugged Asset A (Floating ~9,425 Initial Intercept)', 'color': 'crimson', 'ls': '--'},
    '9kRcWv60rDACzjR': {'label': 'Bugged Asset B (Floating ~3,075 Initial Intercept)', 'color': 'darkorange', 'ls': '-.'}
}

# Isolate the timeline loop to a 24-hour cycle on May 15
for key, meta in target_inverters.items():
    single_asset_day = gen_p2[
        (gen_p2['SOURCE_KEY'] == key) & 
        (gen_p2['DATE_STR'] == target_test_date)
    ].sort_values('DATE_TIME')
    
    plt.plot(single_asset_day['TIME_STR'], single_asset_day['DAILY_YIELD'], 
             color=meta['color'], linestyle=meta['ls'], lw=3, label=meta['label'])

# FIX: Anchor structural locator values securely before overriding text descriptors to block UserWarning messages
ax = plt.gca()
current_ticks = ax.get_xticks()
target_indices = np.arange(0, len(single_asset_day['TIME_STR']), 16) # Safely jump every 4 hours (16 steps of 15-min entries)

ax.set_xticks(target_indices)
ax.set_xticklabels(single_asset_day['TIME_STR'].iloc[target_indices], rotation=45)

# Emphasize the ground zero baseline marker
plt.axhline(0, color='black', lw=1.5, linestyle=':', alpha=0.6, label='Theoretical Ground Zero')

plt.title(f"Visual Proof: Cumulative Daily Yield Reset Anomalies on {target_test_date}", fontsize=14, fontweight='bold')
plt.xlabel("24-Hour Timeline Flow", fontsize=11)
plt.ylabel("Accumulated Energy Reading (DAILY_YIELD - kWh)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='center right', fontsize=11)
plt.tight_layout()
plt.show()

# 🏭 Plant 2: Production-Grade Data Preparation & Feature Engineering Pipeline

This section operationalizes our comprehensive exploratory insights into a robust, automated **Production Data Pipeline**. We transform the highly anomalous raw telemetry into a continuous, high-integrity feature space tailored for leak-free machine learning algorithms.

### 📐 Structural Engineering Rules Applied:
1. **Cross-Sectional Median Consolidation:** Symmetrically aggregates the 22 active sub-assets per timestamp via the mathematical median, successfully smoothing out localized inverter dropouts and shifting communication lag into outer statistical tails.
2. **Deterministic 1:1 Causal Join:** Executes a tight relational inner join matching the clean macro generation baseline directly to the isolated plant weather parameters.
3. **Strategic Metric Censorship:** Deliberately filters out `DAILY_YIELD` and `TOTAL_YIELD`. By completely omitting these unstable indices from our active vector arrays, we permanently neutralize the midnight registry cache leak failures without sacrificing physical current metrics.

In [ ]:
# ==============================================================================
# SECTION 16: PRODUCTION-GRADE DATA PREPARATION PIPELINE (ENTERPRISE LOGGING)
# ==============================================================================
# 1. Initialize Pipeline-Level Clean Logger Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("SolarPipeline")

# Enforce active local path routing checks
gen_path = '../data/raw/Plant_2_Generation_Data.csv' if os.path.exists('../data/raw/Plant_2_Generation_Data.csv') else 'Plant_2_Generation_Data.csv'
weather_path = '../data/raw/Plant_2_Weather_Sensor_Data.csv' if os.path.exists('../data/raw/Plant_2_Weather_Sensor_Data.csv') else 'Plant_2_Weather_Sensor_Data.csv'

logger.info("Initializing raw Plant 2 datasets ingestion matrix...")
gen_raw = pd.read_csv(gen_path)
weather_raw = pd.read_csv(weather_path)

gen_raw['DATE_TIME'] = pd.to_datetime(gen_raw['DATE_TIME'])
weather_raw['DATE_TIME'] = pd.to_datetime(weather_raw['DATE_TIME'])


# 2. Cross-Sectional Median Aggregation
logger.info("Executing cross-sectional median transformations on generation vectors.")
gen_cleansed = gen_raw.groupby('DATE_TIME')[['DC_POWER', 'AC_POWER']].median().reset_index()


# 3. Clean 1:1 Relational Inner Join
logger.info("Merging power generation timelines with isolated atmospheric matrices.")
plant2_master = pd.merge(gen_cleansed, weather_raw, on='DATE_TIME', how='inner')


# 4. Feature Isolation
target_features = ['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'DC_POWER', 'AC_POWER']
plant2_master = plant2_master[target_features]


# 5. Output Verification & Directory Generation Safeguard
output_path = '../data/processed/plant2_master_features.csv'
output_dir = os.path.dirname(output_path) if output_path.startswith('..') else '.'

if output_dir and output_dir != '.' and not os.path.exists(output_dir):
    os.makedirs(output_dir)
    logger.info(f"Target processing directories generated at: {output_dir}")

logger.info(f"Exporting finalized features matrix to: {output_path}")
plant2_master.to_csv(output_path, index=False)


# 6. Pipeline Integrity Sign-Off Report (Keep this structured print for clean visual audit)
print("\n" + "="*80)
print(" PLANT 2 DATA PREPARATION PIPELINE SIGN-OFF REPORT")
print("==============================================================================")
print(f"Total Rows Compiled    : {plant2_master.shape[0]:,} rows")
print(f"Retained Feature Matrix: {list(plant2_master.columns)}")
print(f"Null Integrity Check   : {plant2_master.isnull().sum().sum()} missing records remaining.")
print("==============================================================================")
logger.info("Pipeline complete. Data is highly optimized for leak-free LightGBM modeling!")
print("==============================================================================")

# 🤖 Plant 2: Prospective Time-Series Partitioning & LightGBM Machine Learning Pipeline

This final operational checkpoint implements a rigorous **Chronological Holdout Split** (strictly avoiding random data shuffling to eliminate target leakage) and fits an enterprise-grade **LightGBM Regressor** to project prospective power forecasting vectors into an unseen future timeline.

### 📊 Validation Architecture Strategy:
1. **Historical Training Base (05/15 ~ 06/11):** Establishes the core atmospheric causality matrix mapping $X$ ($Ambient\_Temperature$, $Module\_Temperature$, $Irradiation$) to the physical $DC\_POWER$ generation target.
2. **Prospective Future Test Window (06/12 ~ 06/17):** Functions as a completely blind evaluation timeline, simulating how the pipeline handles real-time streaming operations on unseen future calendar slots.
3. **Enterprise Structured Logging Engine:** Replaces casual terminal prints with a synchronized, production-grade tracking standard to record optimization checkpoints cleanly.

In [ ]:
# ==============================================================================
# SECTION 17: TIME-SERIES PARTITION & LIGHTGBM EVALUATION (ENTERPRISE LOGGING)
# ==============================================================================
# 1. Initialize Standard Pipeline-Level Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("ModelingPipeline")

# Route path configurations safely
processed_path = '../data/processed/plant2_master_features.csv' if os.path.exists('../data/processed/plant2_master_features.csv') else 'plant2_master_features.csv'

logger.info(f"Ingesting preprocessed master features matrix from: {processed_path}")
df_master = pd.read_csv(processed_path)

# Enforce strict datetime data types and extract string trackers
df_master['DATE_TIME'] = pd.to_datetime(df_master['DATE_TIME'])
df_master['DATE_STR'] = df_master['DATE_TIME'].dt.strftime('%Y-%m-%d')


# 2. Feature Isolation (X) and Target Vector (y) Mapping
features = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
X = df_master[features]
y = df_master['DC_POWER']
dates = df_master['DATE_STR']


# 3. Apply Strict Chronological Holdout Split (Zero Leakage Framework)
train_mask = dates <= '2020-06-11'
test_mask  = (dates >= '2020-06-12') & (dates <= '2020-06-17')

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test   = X[test_mask], y[test_mask]


# 4. Pipeline Partition Structure Verification Audit
print("\n" + "="*80)
print(" CHRONOLOGICAL TIMELINE PARTITION INTEGRITY REPORT")
print("==============================================================================")
print(f"Training Knowledge Base (05/15 ~ 06/11) : {X_train.shape[0]:,} rows")
print(f"Evaluation Test Window  (06/12 ~ 06/17) : {X_test.shape[0]:,} rows")
print("==============================================================================\n")


# 5. Fit Enterprise-Grade LightGBM Regressor on Training Baseline
logger.info("Training LightGBM forecasting engine on operational training historical rows...")
model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
model.fit(X_train, y_train)


# 6. Execute Prospective Forecasting Over the Blind Test Window
logger.info("Projecting prospective power vectors over the unseen future test window.")
y_pred = model.predict(X_test)


# 7. Evaluate True Chronological Performance Metrics
rmse_eval = np.sqrt(mean_squared_error(y_test, y_pred))
r2_eval = r2_score(y_test, y_pred)


# 8. Render Final Production Score Sign-Off
print("\n" + "="*80)
print(" FINAL OPERATIONAL PERFORMANCE SIGN-OFF: PLANT 2 FORECASTING REPORT")
print("==============================================================================")
print(f"Plant 2 Future Validation RMSE     : {rmse_eval:.4f} kW")
print(f"Plant 2 Future Validation R2 Score : {r2_eval:.4f} ({r2_eval*100:.2f}% Variance Match)")
print("==============================================================================")
logger.info("Modeling pipeline evaluation sequence completed successfully.")
print("==============================================================================")

# 📊 Plant 2: LightGBM Feature Importance & Causal Auditing Pipeline

This section extracts the **Relative Feature Importance Scores** from our trained LightGBM regressor. By reviewing the architectural split weights, we perform a domain-driven sanity check to ensure the algorithmic decisions align with the physical rules of solar photovoltaic dynamics.

### 📐 Interpretability and Validation Rationale:
* **Physical Causality Verification:** Confirming whether atmospheric parameters (such as $Irradiation$ or $Module\_Temperature$) command the highest structural split weights, matching the ground-truth energy conversion flow.
* **XAI Compliance Sign-Off:** Providing visual and numerical explainability assets necessary for engineering review boards to trust the prospective multi-day forecasting output.

In [ ]:
# ==============================================================================
# SECTION 18: LIGHTGBM FEATURE IMPORTANCE AUDIT (ENTERPRISE LOGGING)
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("ModelingPipeline")
logger.info("Extracting operational feature importance indices from the trained LightGBM model.")

# Extract split importances from the synchronized model instance
importance_values = model.feature_importances_
feature_names = X.columns

# 2. Structure into a Clean Compliance Evaluation DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_values
}).sort_values('Importance', ascending=True)

# 3. Render Horizontal Bar Chart Layout Symmetrically
plt.figure(figsize=(10, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='dodgerblue', edgecolor='darkblue', alpha=0.8)

plt.title("Plant 2: LightGBM Feature Importance (Gain/Split Weights)", fontsize=14, fontweight='bold')
plt.xlabel("Relative Importance Score", fontsize=11)
plt.ylabel("Telemetry Sensor Features", fontsize=11)
plt.grid(True, axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

logger.info("Feature importance canvas rendered successfully. Model interpretability audit complete.")

# 📊 Plant 2: Prospective Forecast Validation Dashboard & Time-Integration Audit

This section executes our final **Prospective Forecasting Validation Dashboard**. We translate instant 15-minute power vectors ($kW$) into aggregated daily energy capacities ($kWh$) through explicit mathematical time-integration layer. 

$$\text{Energy Space (kWh)} = \text{Power Space (kW)} \times \left( \frac{15 \text{ Minutes}}{60 \text{ Minutes}} \right)$$

### 📐 Valuation Diagnostics Applied:
1. **Dynamic Inversion Trajectory Layer:** Rather than mapping un-cleansed registers, we dynamically derive the physical $AC/DC$ conversion efficiency footprint from the historical training split to transform the LightGBM $DC$ predictions into pure $AC$ energy metrics safely.
2. **Side-by-Side Volume Audit (Chart 1):** Compares the daily absolute energy capacity generated versus the algorithmic predictions over the blind chronological validation window.
3. **Cumulative Momentum Tracking (Chart 2):** Evaluates the long-term compounding error ("Chiri-tsumo" parallel shift effect) by mapping ongoing trajectory lines to ensure model safety margins.

In [ ]:
# ==============================================================================
# SECTION 19: PROSPECTIVE FORECAST VALIDATION DASHBOARD (ENTERPRISE LOGGING)
# ==============================================================================
# # 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("ModelingPipeline")
logger.info("Initializing mathematical time-integration layer over the prospective test splits.")

# Verify and Execute Prediction Vectors (Using variables from Pipeline B)
if 'model' not in locals() or 'X_test' not in locals():
    logger.error("Required variables 'model' or 'X_test' missing in the current namespace layout.")
    raise NameError("Please execute the previous LightGBM modeling block to initialize your variables.")

# Generate core predictions over the out-of-time validation test split
y_pred_dc = model.predict(X_test)

# 2. Reconstruct Evaluation Matrix for Detailed Aggregations
validation_df = pd.DataFrame({
    'DATE_TIME': df_master.loc[test_mask, 'DATE_TIME'],
    'DATE_STR': df_master.loc[test_mask, 'DATE_STR'],
    'DC_ACTUAL': y_test.values,
    'DC_PREDICTED': y_pred_dc,
    'AC_ACTUAL': df_master.loc[test_mask, 'AC_POWER'].values
})

# Calculate the physical Plant 2 conversion coefficient dynamically from the training matrix
train_clean = df_master[train_mask]
conversion_ratio = (train_clean['AC_POWER'] / train_clean['DC_POWER']).replace([np.inf, -np.inf], np.nan).dropna().mean()

# Convert predicted DC vectors into AC metrics using the dynamic conversion rule
validation_df['AC_PREDICTED'] = validation_df['DC_PREDICTED'] * conversion_ratio

# 3. Time-Integration Layer: Convert 15-minute power spikes (kW) into daily energy capacities (kWh)
validation_df['YIELD_ACTUAL_15MIN'] = validation_df['AC_ACTUAL'] * (15.0 / 60.0)
validation_df['YIELD_PRED_15MIN'] = validation_df['AC_PREDICTED'] * (15.0 / 60.0)

# Group by calendar date to calculate strict cumulative daily summaries
daily_yield_summary = validation_df.groupby('DATE_STR').agg({
    'YIELD_ACTUAL_15MIN': 'sum',
    'YIELD_PRED_15MIN': 'sum'
}).reset_index()

# Rename columns for enterprise clarity
daily_yield_summary.columns = ['DATE_STR', 'DAILY_YIELD_ACTUAL', 'DAILY_YIELD_PREDICTED']

# Calculate ongoing cumulative total trajectories across the 6-day test split
daily_yield_summary['TOTAL_YIELD_ACTUAL'] = daily_yield_summary['DAILY_YIELD_ACTUAL'].cumsum()
daily_yield_summary['TOTAL_YIELD_PREDICTED'] = daily_yield_summary['DAILY_YIELD_PREDICTED'].cumsum()


# ------------------------------------------------------------------------------
# CHART 1: DAILY ACCUMULATED YIELD COMPARISON (ACTUAL VS PREDICTED BARS)
# ------------------------------------------------------------------------------
logger.info("Rendering Chart 1: side-by-side daily accumulated energy yield validation bars.")
plt.figure(figsize=(14, 6))
x_indices = np.arange(len(daily_yield_summary))
bar_width = 0.35

# Plot side-by-side comparative blocks
plt.bar(x_indices - bar_width/2, daily_yield_summary['DAILY_YIELD_ACTUAL'], 
        width=bar_width, color='royalblue', edgecolor='darkblue', alpha=0.8, label='Actual Daily Yield (Real-World Data)')
plt.bar(x_indices + bar_width/2, daily_yield_summary['DAILY_YIELD_PREDICTED'], 
        width=bar_width, color='orange', edgecolor='darkorange', alpha=0.8, label='Predicted Daily Yield (LightGBM Forecast)')

# FIX: Explicitly set the ticks array index alignment before mapping string text elements to block UserWarning diagnostics
plt.gca().set_xticks(x_indices)
plt.gca().set_xticklabels(daily_yield_summary['DATE_STR'], rotation=30, fontsize=9)

plt.title("Plant 2: 6-Day Daily Energy Yield Validation (Actual vs Predicted Volume)", fontsize=14, fontweight='bold')
plt.xlabel("Validation Timeline Target (Unseen Future Rows)", fontsize=11)
plt.ylabel("Energy Yield Volume (kWh)", fontsize=11)
plt.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------------------
# CHART 2: LIFETIME TOTAL YIELD CONTINUOUS ESCALATION (ACTUAL VS PREDICTED LINE)
# ------------------------------------------------------------------------------
logger.info("Rendering Chart 2: total cumulative energy escalation trajectory lines.")
plt.figure(figsize=(14, 6))

# Plot actual track vs predicted trajectory over the continuous timeline
plt.plot(daily_yield_summary['DATE_STR'], daily_yield_summary['TOTAL_YIELD_ACTUAL'], 
         color='royalblue', lw=3, marker='o', ms=7, label='Actual Cumulative Growth Track')
plt.plot(daily_yield_summary['DATE_STR'], daily_yield_summary['TOTAL_YIELD_PREDICTED'], 
         color='orange', lw=3, linestyle='--', marker='s', ms=7, label='Predicted Cumulative Growth Track')

# FIX: Anchor the explicit indices array positions to enforce index stability under shared subplots configuration
plt.gca().set_xticks(x_indices)
plt.gca().set_xticklabels(daily_yield_summary['DATE_STR'], rotation=30, fontsize=9)

plt.title("Plant 2: 6-Day Total Cumulative Energy Escalation (Actual vs Predicted Trajectory)", fontsize=14, fontweight='bold')
plt.xlabel("Validation Timeline Target (Unseen Future Rows)", fontsize=11)
plt.ylabel("Total Consolidated Energy Volume (Cumulative kWh)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

logger.info("Validation metrics visualizations rendered successfully. Operational audit sequence closed.")